In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

InMemorySaver stores data in RAM, it is only used for testing purpose, in production langgraph provides other libraries to store STATES in the DBs

In [3]:
load_dotenv()

llm = ChatOpenAI()

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!",
 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd63-6851-6f93-8002-10d1f9dcddd8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-31T06:18:45.360833+00:00', parent_config={'con

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd63-6851-6f93-8002-10d1f9dcddd8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-31T06:18:45.360833+00:00', parent_config={'co

#### ThreadID-2

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the pasta go to the party alone? \n\nBecause it didn't want anyBODY to spoil its sauce!",
 'explanation': 'This joke plays on the pun of "body" sounding like "buddy". In this case, the pasta went to the party alone because it didn\'t want any "body" (person) to spoil its sauce. The humor is derived from the unexpected twist in the wordplay, as we initially assume the pasta doesn\'t want any human to spoil its meal, but it turns out to be a pun based on its sauce.'}

In [17]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the party alone? \n\nBecause it didn't want anyBODY to spoil its sauce!", 'explanation': 'This joke plays on the pun of "body" sounding like "buddy". In this case, the pasta went to the party alone because it didn\'t want any "body" (person) to spoil its sauce. The humor is derived from the unexpected twist in the wordplay, as we initially assume the pasta doesn\'t want any human to spoil its meal, but it turns out to be a pun based on its sauce.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd63-9a1b-632f-8002-e96cdbf926b5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '2'}, created_at='2025-07-31T06:18:50.581278+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd63-849f-6982-8001-50c81244f3e3'}}, tasks=(), interrupts=())

In [18]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pasta go to the party alone? \n\nBecause it didn't want anyBODY to spoil its sauce!", 'explanation': 'This joke plays on the pun of "body" sounding like "buddy". In this case, the pasta went to the party alone because it didn\'t want any "body" (person) to spoil its sauce. The humor is derived from the unexpected twist in the wordplay, as we initially assume the pasta doesn\'t want any human to spoil its meal, but it turns out to be a pun based on its sauce.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd63-9a1b-632f-8002-e96cdbf926b5'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '2'}, created_at='2025-07-31T06:18:50.581278+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd63-849f-6982-8001-50c81244f3e3'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the pa

### Time Travel

travelling between diffrent checkpoints is possible through the checkpoint ID.

In [15]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06dd63-6851-6f93-8002-10d1f9dcddd8"}})

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f06dd63-6851-6f93-8002-10d1f9dcddd8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-31T06:18:45.360833+00:00', parent_config={'configurable': {'thread_

In [20]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06dd63-6851-6f93-8002-10d1f9dcddd8"}})

{'topic': 'pizza',
 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!",
 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd63-6851-6f93-8002-10d1f9dcddd8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-31T06:18:45.360833+00:00', parent_config={'co

In [23]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06dd63-6851-6f93-8002-10d1f9dcddd8", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f06dd6b-d97a-658c-8003-18312efac63d'}}

In [24]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd6b-d97a-658c-8003-18312efac63d'}}, metadata={'source': 'update', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-31T06:22:31.974644+00:00', parent_config={

In [ ]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f06dd63-6851-6f93-8002-10d1f9dcddd8"}})

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': "Why did the pizza go to the therapist?\n\nBecause it had too many toppings and couldn't hold it all together!", 'explanation': 'This joke plays on the idea of a pizza seeking therapy as a person would to deal with emotional issues. The punchline reveals that the pizza\'s reason for seeking therapy is because it had too many toppings, causing it to not be able to hold itself together. This is a play on words, as "hold it all together" can refer to both the physical structure of the pizza with too many toppings as well as the pizza\'s mental/emotional state needing help to cope with the overwhelming situation. Overall, the joke is meant to be light-hearted and humorous.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06dd6b-d97a-658c-8003-18312efac63d'}}, metadata={'source': 'update', 'step': 3, 'parents': {}, 'thread_id': '1'}, created_at='2025-07-31T06:22:31.974644+00:00', parent_config={